In [ ]:
!pip install torch transformers langchain PyMuPDF faiss-cpu tqdm
!pip install sentence-transformers

In [ ]:
import re
from tqdm import tqdm
from langchain.vectorstores import FAISS
from langchain.docstore.document import Document
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import fitz  
import csv
import json

# Step 1: Extract Text and Headings from PDF
pdf_file_path = "/kaggle/input/casml-generative-ai-hackathon/Dataset_RAG (1)/book.pdf"  

def extract_sections_and_text(pdf_path):
    extracted_content = []
    current_section = None

    with fitz.open(pdf_path) as doc:
        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            text = page.get_text("text")

            for line in text.split("\n"):
                if is_heading(line):
                    current_section = format_section_name(line)
                extracted_content.append({
                    "text": line.strip(),
                    "section": current_section,
                    "page": page_num + 1
                })
    return extracted_content

def is_heading(line):
    return bool(re.match(r"^\s*\d*\.\d*\s+.+$", line))

def format_section_name(line):
    return line.strip().lower().replace(" ", "_").replace(".", "/")

extracted_content = extract_sections_and_text(pdf_file_path)

# Step 2: Process and Chunk Text
tokenizer = AutoTokenizer.from_pretrained("thenlper/gte-large")
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=1500,
    chunk_overlap=500,  
    add_start_index=True,
    strip_whitespace=True,
)

def process_text_with_sections(extracted_content):
    docs_processed = []
    for item in extracted_content:
        chunks = text_splitter.split_text(item["text"])
        for chunk in chunks:
            docs_processed.append(Document(
                page_content=chunk,
                metadata={
                    "section": item["section"],
                    "page": item["page"],
                }
            ))
    return docs_processed

docs_processed = process_text_with_sections(extracted_content)

# Step 3: Create FAISS Vector Store
embedding_model = HuggingFaceEmbeddings(model_name="thenlper/gte-large")
vectordb = FAISS.from_documents(docs_processed, embedding=embedding_model, distance_strategy="cosine")
print('vectordb saved')

In [ ]:
# Step 4: Define RetrieverTool
class RetrieverTool:
    def _init_(self, vectordb):
        self.vectordb = vectordb

    def forward(self, query: str, k=7):
        docs = self.vectordb.similarity_search(query, k=k)
        return docs

retriever_tool = RetrieverTool(vectordb)

# Step 5: Retrieve Context with Targeting
def retrieve_context(query, collection, n_results=5):
    content_queries = [
        "scientific method psychology methodology",
        "research methods psychology definition",
        "scientific method steps psychology",
        "psychology research methodology principles",
        query
    ]

    all_contexts = []
    all_pages = set()

    for content_query in content_queries:
        results = collection.similarity_search(content_query, k=n_results)
        for doc in results:
            page = doc.metadata["page"]
            if 40 <= page <= 60 and ("scientific method" in doc.page_content.lower() or "research method" in doc.page_content.lower()):
                all_contexts.append(f"[Page {page}]: {doc.page_content}")
                all_pages.add(page)

    return {
        'context': ' '.join(all_contexts[:3]),  
        'pages': sorted(list(all_pages))
    }


# Step 6: Generate References
def generate_references(docs):
    sections = list({doc.metadata["section"] for doc in docs if doc.metadata["section"]})
    pages = sorted({doc.metadata["page"] for doc in docs})
    return {
        "sections": sections,
        "pages": pages
    }

# Step 7: Load Local LLM for Inference
llm_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct", trust_remote_code=True).to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
llm_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct", trust_remote_code=True)

def generate_response(prompt: str):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    inputs = llm_tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    outputs = llm_model.generate(**inputs, max_new_tokens=256)
    return llm_tokenizer.decode(outputs[0], skip_special_tokens=True)

# Step 8: Define Precision Submission Function
def submissionfunc(query_id, query, collection):
    concept_results = collection.similarity_search(f"definition explanation {query}", k=2)
    detail_results = collection.similarity_search(f"example application {query}", k=2)

    contexts = []
    pages = set()

    for results in [concept_results, detail_results]:
        for doc in results:
            contexts.append(doc.page_content)
            pages.add(doc.metadata["page"])

    context = ' '.join(contexts[:2])
    answer = generate_response(f"Answer the question: {query}\nContext: {context}")
    references = {
        'sections': [doc.metadata.get("section", "unknown") for doc in concept_results],
        'pages': list(sorted(pages))[:3]
    }

    return {
        'ID': query_id,
        'context': context,
        'answer': answer.strip(),
        'references': json.dumps(references, ensure_ascii=False)
    }

# Step 9: Save Results to CSV
def save_results_to_csv(results, output_file="submission.csv"):
    with open(output_file, mode="w", newline="", encoding="utf-8") as csvfile:
        fieldnames = ["ID", "context", "answer", "references"]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

        writer.writeheader()
        for result in results:
            writer.writerow({
                "ID": int(result["ID"]),
                "context": result["context"],
                "answer": result["answer"].strip(),
                "references": json.dumps(result["references"], ensure_ascii=False),
            })

# Step 10: Load Queries and Generate Results
queries_file = "/kaggle/input/casml-generative-ai-hackathon/Dataset_RAG (1)/queries.json" 

with open(queries_file, "r", encoding="utf-8") as f:
    queries = json.load(f)

results = []
for query in tqdm(queries):
    query_id = query.get("query_id")
    question = query.get("question")
    if query_id is not None and question:
        result = submissionfunc(query_id, question, vectordb)
        results.append(result)

save_results_to_csv(results)
print("Submission saved")